# H SC 790 · Lab 1 · Build a Reportable-Disease Registry

**Week 2.** Due Thursday, February 4, 11:59 PM · 30 points.

**What earns full credit**
- Runs end to end from a fresh runtime
- Model is in third normal form, or you say why not
- Interpretation cites your own output
- AI-use note included

**Rules.** Everything here is synthetic. Never load protected health information or confidential employer data into Colab or GitHub. The lab is finished when it runs top to bottom from a fresh runtime (Runtime → Restart session and run all).

The data dictionary for this lab is the README in the lab's folder of the course repository.

Cells marked `# YOUR CODE HERE` and *Your answer here.* are yours to complete. Add cells freely.

In [ ]:
# Setup: run this cell first, every time.
import os, json, urllib.request
import numpy as np
import pandas as pd

# The course data, read straight from the course repository. Never from your own machine.
COURSE_URL = os.environ.get("HSC790_DATA", "https://raw.githubusercontent.com/lmlevine/hsc790-spr27/main/data/")

def read_text(rel):
    """Text of one file in the course data folder, for example read_text('SNAPSHOT.txt')."""
    return urllib.request.urlopen(COURSE_URL + rel).read().decode()

print("Data snapshot:", read_text("SNAPSHOT.txt").strip())

`lab01/raw_reports.csv` is every case report the Department of Public Health received from January 1 through March 31, 2026, exported as one wide table. Turn it into a normalized registry in DuckDB with keys, constraints, and a data dictionary. The schema you build is the store for Labs 2 through 7.

In [ ]:
import duckdb
con = duckdb.connect()
con.sql(f"CREATE TABLE raw_reports AS SELECT * FROM read_csv_auto('{COURSE_URL}lab01/raw_reports.csv')")
con.sql("SELECT * FROM raw_reports LIMIT 3").df()

## 1. Profile the raw extract

Count rows, distinct patients, distinct facilities, and nulls per column. Say what one row is. Look closely at the columns you will turn into their own tables: do the values that should repeat exactly really repeat exactly?

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

**What is one row? What did profiling show that your design has to handle?**

*Your answer here.*

## 2. Draw the logical model

Identify entities, keys, and cardinalities. Sketch the ER diagram here (text is fine) or attach an image.

*Your answer here.*

## 3. Write the DDL

Primary keys, foreign keys, NOT NULL, UNIQUE, and at least two CHECK constraints.

In [ ]:
# YOUR CODE HERE

## 4. Load and transform

Populate each table from the raw extract with `INSERT … SELECT`. Start with the slide's statement for `facility` and see what happens.

In [ ]:
try:
    con.sql("""
      INSERT INTO facility
      SELECT DISTINCT facility_id, TRIM(facility_name),
             facility_type, clia_number
      FROM raw_reports
      WHERE facility_id IS NOT NULL
    """)
    print("loaded")
except duckdb.ConstraintException as e:
    print("failed:", e)

If it failed, work out why from your profile in step 1, and fix the load, not the constraint. Then load the remaining tables. Split `race` into the child table.

In [ ]:
# YOUR CODE HERE

## 5. Prove the constraints work

Attempt three inserts that should fail, and show that each one does. The first is the slide's. Then route the source rows that violate your rules to a quarantine table, with the reason, and load the rest.

In [ ]:
# a test that SHOULD fail: report before specimen
try:
    con.sql("""INSERT INTO case_report VALUES
      (999999, 1, 'PERT', 10, NULL,
       DATE '2026-03-10', NULL, DATE '2026-03-01',
       'confirmed')""")
    print('PROBLEM: bad row was accepted')
except duckdb.ConstraintException as e:
    print('rejected as designed:', e)

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

## 6. Write the data dictionary

One entry per column: definition, type, permissible values, required, source, steward. Save it as a CSV in your lab folder.

In [ ]:
# YOUR CODE HERE

## Interpretation

Cite your own output in each answer.

**1. How many source rows did your constraints reject? Which rule rejected the most, and is that rule too strict?**

*Your answer here.*

**2. Where did you depart from third normal form? Name the table and give the reason.**

*Your answer here.*

**3. Which definition was hardest to write? Quote your dictionary entry and say what argument it settles.**

*Your answer here.*

**4. Who would be affected by your strictest rule? Which reporters or populations would lose cases if this ran in production?**

*Your answer here.*

## AI-use note (required)

Say which AI tools you used, for what, and one thing you checked or corrected in their output. "None" is an acceptable answer.

*Your note here.*

## Submit

1. Runtime → Restart session and run all. Fix anything that fails.
2. File → Save a copy in GitHub, into `labs/lab01/` of your course repository.